# Generate dim_account
Creates the account dimension and writes data/dim_account.csv.
This notebook reads existing dimension CSVs so foreign keys are sampled only from valid surrogate keys.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
COUNT = 3000
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'README.md').exists() and (candidate / '01_data').exists():
            return candidate
    return current

project_root = find_project_root()
data_dir = project_root / '01_data' / 'raw'

dim_customer = pd.read_csv(data_dir / 'dim_customer.csv')
dim_product = pd.read_csv(data_dir / 'dim_product.csv')
dim_branch = pd.read_csv(data_dir / 'dim_branch.csv')
dim_currency = pd.read_csv(data_dir / 'dim_currency.csv')

customer_keys = dim_customer['customer_key'].to_numpy()
product_keys = dim_product['product_key'].to_numpy()
branch_keys = dim_branch['branch_key'].to_numpy()
currency_keys = dim_currency['currency_key'].to_numpy()

dim_account = pd.DataFrame({
    'account_key': np.arange(1, COUNT + 1, dtype=np.int32),
    'account_number': [f'AC{10000000 + i}' for i in range(COUNT)],
    'customer_key': rng.choice(customer_keys, size=COUNT, replace=True),
    'product_key': rng.choice(product_keys, size=COUNT, replace=True),
    'branch_key': rng.choice(branch_keys, size=COUNT, replace=True),
    'currency_key': rng.choice(currency_keys, size=COUNT, replace=True),
    'opened_date': pd.to_datetime(rng.choice(pd.date_range('2019-01-01', '2026-12-31'), size=COUNT)),
    'status': rng.choice(['Active', 'Dormant', 'Closed'], size=COUNT, p=[0.86, 0.10, 0.04]),
})

data_dir.mkdir(parents=True, exist_ok=True)
output_path = data_dir / 'dim_account.csv'
dim_account.to_csv(output_path, index=False)

print(f'Rows: {len(dim_account)}')
print(f'Wrote: {output_path}')
dim_account.head()